In [103]:
# ============================================================
# Item-Item Co-Purchase Graph in TensorFlow 2
# End-to-end version with:
# - tf.data positive dataset
# - per-batch negative sampling
# - optional false-negative filtering
# - model.fit() style training
# - separate validation metrics at epoch end
# - manual weighted BCE for tf.keras compatibility
# ============================================================

import random
from typing import Dict, Set, Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import sparse

# ------------------------------------------------------------
# 1) Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def build_copurchase_graph(
		df: pd.DataFrame,
		order_col: str = "order_id",
		item_col: str = "product_id",
		min_item_freq: int = 2,
		min_items_per_order: int = 5,
		max_items_per_order: int = 20,
		weight_mode: str = "lift",  # "cosine", "lift", "lift_shrink", "ppmi", "confidence"
		topk_neighbors: int = 200,
		shrink_alpha: float = 10.0,
		dtype=np.float32,
):
	work = df[[order_col, item_col]].dropna().copy()
	work[item_col] = work[item_col].astype(str)
	if work.empty:
		raise ValueError("No data after dropna")

	order_sizes = work.groupby(order_col, sort=False)[item_col].size()
	valid_orders = order_sizes[
		(order_sizes >= min_items_per_order) & (order_sizes <= max_items_per_order)
		].index
	work = work[work[order_col].isin(valid_orders)]

	if work.empty:
		raise ValueError("No data left after basket-size filtering")

	item_freq_series = work.groupby(item_col, sort=False)[order_col].size()
	valid_items = item_freq_series[item_freq_series >= min_item_freq].index
	work = work[work[item_col].isin(valid_items)]

	if work.empty:
		raise ValueError("No data left after min_item_freq filtering")

	item_codes, item_uniques = pd.factorize(work[item_col], sort=False)
	order_codes, order_uniques = pd.factorize(work[order_col], sort=False)

	n_orders = len(order_uniques)
	n_items = len(item_uniques)

	data = np.ones(len(work), dtype=dtype)
	X = sparse.coo_matrix(
		(data, (order_codes.astype(np.int32), item_codes.astype(np.int32))),
		shape=(n_orders, n_items),
		dtype=dtype,
	).tocsr()

	item_freq = np.asarray(X.sum(axis=0)).ravel().astype(np.float64)

	C = (X.T @ X).tocsr()
	C.setdiag(0)
	C.eliminate_zeros()

	rows = []
	total_orders = float(n_orders)

	indptr = C.indptr
	indices = C.indices
	counts = C.data.astype(np.float64, copy=False)

	supported_modes = {"cosine", "lift", "lift_shrink", "ppmi", "confidence"}
	if weight_mode not in supported_modes:
		raise ValueError(f"weight_mode must be one of: {sorted(supported_modes)}")

	for i in range(n_items):
		start, end = indptr[i], indptr[i + 1]
		if start == end:
			continue

		nbr_idx = indices[start:end]
		c_ij = counts[start:end]

		n_i = item_freq[i]
		n_j = item_freq[nbr_idx]

		# compute all metrics
		cosine = c_ij / np.sqrt(np.maximum(n_i * n_j, 1.0))
		lift = (c_ij * total_orders) / np.maximum(n_i * n_j, 1.0)

		shrink = c_ij / (c_ij + float(shrink_alpha))
		lift_shrink = lift * shrink

		p_ij = c_ij / total_orders
		p_i = n_i / total_orders
		p_j = n_j / total_orders
		pmi = np.log((p_ij + 1e-12) / (p_i * p_j + 1e-12))
		ppmi = np.maximum(pmi, 0.0)

		confidence = c_ij / np.maximum(n_i, 1.0)

		metric_map = {
			"cosine": cosine,
			"lift": lift,
			"lift_shrink": lift_shrink,
			"ppmi": ppmi,
			"confidence": confidence,
		}
		rank_w = metric_map[weight_mode]

		if len(rank_w) > topk_neighbors:
			top_local = np.argpartition(-rank_w, topk_neighbors - 1)[:topk_neighbors]

			nbr_idx = nbr_idx[top_local]
			c_ij = c_ij[top_local]

			cosine = cosine[top_local]
			lift = lift[top_local]
			shrink = shrink[top_local]
			lift_shrink = lift_shrink[top_local]
			ppmi = ppmi[top_local]
			confidence = confidence[top_local]
			rank_w = rank_w[top_local]

		order = np.lexsort((-c_ij, -rank_w))[::-1]

		src_item = item_uniques[i]
		for k in order:
			rows.append(
				(
					src_item,
					item_uniques[nbr_idx[k]],
					float(rank_w[k]),  # selected ranking weight
					float(cosine[k]),
					float(lift[k]),
					float(shrink[k]),
					float(lift_shrink[k]),
					float(ppmi[k]),
					float(confidence[k]),
					int(c_ij[k]),
				)
			)

	if not rows:
		raise ValueError("No graph edges created after sparse multiplication")

	edges_df = pd.DataFrame(
		rows,
		columns=[
			"src_item",
			"dst_item",
			"weight",  # active ranking weight based on weight_mode
			"cosine",
			"lift",
			"shrink",
			"lift_shrink",
			"ppmi",
			"confidence",
			"co-occurance",
		],
	)

	item_freq_dict = pd.Series(item_freq, index=item_uniques).astype(int).to_dict()
	return edges_df, item_freq_dict


# ------------------------------------------------------------
# 3) Train/validation split
# ------------------------------------------------------------
def train_val_split_edges(
		edges_df: pd.DataFrame,
		val_frac: float = 0.2,
		seed: int = SEED,
):
	"""
    Random edge split.
    For stronger evaluation, later you may want a basket-level or time-based split.
    """
	if not 0.0 < val_frac < 1.0:
		raise ValueError("val_frac must be between 0 and 1")

	n = len(edges_df)
	if n < 2:
		raise ValueError("Need at least 2 edges for train/validation split")

	rng = np.random.default_rng(seed)
	idx = np.arange(n)
	rng.shuffle(idx)

	n_val = max(1, int(n * val_frac))
	val_idx = idx[:n_val]
	train_idx = idx[n_val:]

	if len(train_idx) == 0:
		raise ValueError("Train split is empty after split")

	train_edges = edges_df.iloc[train_idx].reset_index(drop=True)
	val_edges = edges_df.iloc[val_idx].reset_index(drop=True)
	return train_edges, val_edges


# ------------------------------------------------------------
# 4) Vocabulary + arrays
# ------------------------------------------------------------
def build_item_vocab(edges_df: pd.DataFrame):
	items = pd.Index(
		pd.concat([edges_df["src_item"], edges_df["dst_item"]]).astype(str).unique()
	)
	item2idx = {item: idx for idx, item in enumerate(items)}
	idx2item = {idx: item for item, idx in item2idx.items()}
	return item2idx, idx2item


def edges_to_training_arrays(edges_df: pd.DataFrame, item2idx: dict):
	src = edges_df["src_item"].map(item2idx).to_numpy(dtype=np.int32)
	dst = edges_df["dst_item"].map(item2idx).to_numpy(dtype=np.int32)
	wgt = edges_df["weight"].to_numpy(dtype=np.float32)

	if len(wgt) > 0:
		wgt = wgt / (wgt.mean() + 1e-8)

	return src, dst, wgt


# ------------------------------------------------------------
# 5) Negative sampling distribution
# ------------------------------------------------------------
def build_negative_sampling_distribution(
		edges_df: pd.DataFrame,
		item2idx: dict,
		mode: str = "uniform",  # "popular" or "uniform"
		power: float = 0.75,
):
	if mode == "uniform":
		probs = np.ones(len(item2idx), dtype=np.float64)
		probs = probs / probs.sum()
		return probs.astype(np.float32)

	if mode == "popular":
		freq = edges_df["dst_item"].value_counts()
		probs = np.ones(len(item2idx), dtype=np.float64)

		for item, count in freq.items():
			probs[item2idx[item]] = float(count) ** power

		probs = probs / probs.sum()
		return probs.astype(np.float32)

	raise ValueError("mode must be one of: 'popular', 'uniform'")


# ------------------------------------------------------------
# 6) Optional false-negative filtering structures
# ------------------------------------------------------------
def build_positive_neighbor_sets(edges_df: pd.DataFrame, item2idx: dict) -> Dict[int, Set[int]]:
	neighbor_sets = {}
	for src_item, grp in edges_df.groupby("src_item"):
		src_idx = item2idx[src_item]
		dst_set = set(grp["dst_item"].map(item2idx).tolist())
		neighbor_sets[src_idx] = dst_set
	return neighbor_sets


# ------------------------------------------------------------
# 7) tf.data positive-only dataset
# ------------------------------------------------------------
def make_positive_tf_dataset(
		edges_df: pd.DataFrame,
		item2idx: dict,
		batch_size: int = 4096,
		shuffle: bool = True,
):
	pos_src, pos_dst, pos_wgt = edges_to_training_arrays(edges_df, item2idx)

	ds = tf.data.Dataset.from_tensor_slices({
		"src_item": pos_src.astype(np.int32),
		"pos_dst_item": pos_dst.astype(np.int32),
		"pos_weight": pos_wgt.astype(np.float32),
	})

	if shuffle:
		ds = ds.shuffle(
			buffer_size=min(len(pos_src), 100000),
			reshuffle_each_iteration=True,
			seed=SEED,
		)

	ds = ds.batch(batch_size, drop_remainder=False)
	ds = ds.prefetch(tf.data.AUTOTUNE)
	return ds


# ------------------------------------------------------------
# 8) Base scoring model
# ------------------------------------------------------------
class ItemItemEncoder(tf.keras.Model):
	def __init__(self, num_items: int, embedding_dim: int = 64, l2_reg: float = 1e-6):
		super().__init__()
		self.item_embedding = tf.keras.layers.Embedding(
			input_dim=num_items,
			output_dim=embedding_dim,
			embeddings_initializer="glorot_uniform",
			embeddings_regularizer=tf.keras.regularizers.l2(l2_reg),
			name="item_embedding",
		)
		self.item_bias = tf.keras.layers.Embedding(
			input_dim=num_items,
			output_dim=1,
			embeddings_initializer="zeros",
			name="item_bias",
		)
		self.item_embedding.build((None,))
		self.item_bias.build((None,))

	def call(self, inputs, training=False):
		src = inputs["src_item"]  # [N]
		dst = inputs["dst_item"]  # [N]

		src_emb = self.item_embedding(src)  # [N, D]
		dst_emb = self.item_embedding(dst)  # [N, D]
		dst_bias = self.item_bias(dst)  # [N, 1]

		dot = tf.reduce_sum(src_emb * dst_emb, axis=-1, keepdims=True)  # [N, 1]
		logits = dot + dst_bias  # [N, 1]
		return logits


# ------------------------------------------------------------
# 9) Training wrapper with custom train_step + test_step
# ------------------------------------------------------------
class BatchNegativeSamplingModel(tf.keras.Model):
	def __init__(
			self,
			scorer: ItemItemEncoder,
			num_items: int,
			neg_probs: Optional[np.ndarray] = None,
			negatives_per_positive: int = 3,
			positive_neighbor_sets: Optional[Dict[int, Set[int]]] = None,
			filter_false_negatives: bool = False,
			max_resample_attempts: int = 3,
			loss_type: str = "sampled_softmax",
			num_sampled: Optional[int] = None,
			metric_topk: int = 10,
			**kwargs,
	):
		super().__init__(**kwargs)
		self.scorer = scorer
		self.num_items = int(num_items)
		self.negatives_per_positive = int(negatives_per_positive)
		self.filter_false_negatives = bool(filter_false_negatives)
		self.max_resample_attempts = int(max_resample_attempts)

		supported_losses = {"bce", "sampled_softmax"}
		if loss_type not in supported_losses:
			raise ValueError(f"loss_type must be one of: {sorted(supported_losses)}")
		self.loss_type = loss_type
		if metric_topk <= 0:
			raise ValueError("metric_topk must be positive")
		self.metric_topk = int(metric_topk)
		self.metric_name = "auc" if self.loss_type == "bce" else f"top{self.metric_topk}_acc"

		if neg_probs is None:
			neg_probs = np.ones(self.num_items, dtype=np.float32)
		neg_probs = np.asarray(neg_probs, dtype=np.float32)
		neg_probs = neg_probs / neg_probs.sum()
		self.neg_probs_np = neg_probs
		self.neg_logits = tf.constant(np.log(neg_probs[None, :] + 1e-12), dtype=tf.float32)

		max_num_sampled = max(1, self.num_items - 1)
		requested_num_sampled = num_sampled if num_sampled is not None else min(64, max_num_sampled)
		self.num_sampled = int(min(max(1, requested_num_sampled), max_num_sampled))

		self.positive_neighbor_sets = positive_neighbor_sets or {}

		# training metrics
		self.train_loss_tracker = tf.keras.metrics.Mean(name="loss")
		if self.loss_type == "bce":
			self.train_metric = tf.keras.metrics.AUC(name="auc")
			self.val_metric = tf.keras.metrics.AUC(name="val_auc")
		else:
			k = min(self.metric_topk, self.num_items)
			self.train_metric = tf.keras.metrics.SparseTopKCategoricalAccuracy(
				k=k,
				name=f"top{k}_acc",
			)
			self.val_metric = tf.keras.metrics.SparseTopKCategoricalAccuracy(
				k=k,
				name=f"val_top{k}_acc",
			)

		# validation/test metrics
		self.val_loss_tracker = tf.keras.metrics.Mean(name="val_loss")

	@property
	def metrics(self):
		# Keras resets these at the start of each epoch/eval phase
		return [
			self.train_loss_tracker,
			self.train_metric,
			self.val_loss_tracker,
			self.val_metric,
		]

	def call(self, inputs, training=False):
		return self.scorer(inputs, training=training)

	def _sample_negatives_basic(self, src):
		batch_size_tensor = tf.shape(src)[0]
		neg_dst = tf.random.categorical(
			logits=self.neg_logits,
			num_samples=batch_size_tensor * self.negatives_per_positive,
		)
		neg_dst = tf.reshape(neg_dst, [-1])
		neg_dst = tf.cast(neg_dst, tf.int32)
		neg_src = tf.repeat(src, repeats=self.negatives_per_positive)
		return neg_src, neg_dst

	def _sample_negatives_filtered_numpy(self, src_np: np.ndarray, pos_dst_np: np.ndarray):
		bsz = len(src_np)
		total = bsz * self.negatives_per_positive

		neg_src = np.repeat(src_np, self.negatives_per_positive).astype(np.int32)
		neg_dst = np.random.choice(
			self.num_items,
			size=total,
			replace=True,
			p=self.neg_probs_np,
		).astype(np.int32)

		repeated_pos_dst = np.repeat(pos_dst_np, self.negatives_per_positive)

		for i in range(total):
			s = int(neg_src[i])
			true_d = int(repeated_pos_dst[i])
			banned = self.positive_neighbor_sets.get(s, set())

			attempts = 0
			while (neg_dst[i] == true_d or neg_dst[i] in banned) and attempts < self.max_resample_attempts:
				neg_dst[i] = np.random.choice(
					self.num_items,
					size=1,
					replace=True,
					p=self.neg_probs_np,
				)[0]
				attempts += 1

		return neg_src, neg_dst

	def _make_pos_neg_batch(self, src, pos_dst, pos_w):
		if self.filter_false_negatives:
			neg_src, neg_dst = tf.numpy_function(
				func=self._sample_negatives_filtered_numpy,
				inp=[src, pos_dst],
				Tout=[tf.int32, tf.int32],
			)
			neg_src.set_shape([None])
			neg_dst.set_shape([None])
		else:
			neg_src, neg_dst = self._sample_negatives_basic(src)

		all_src = tf.concat([src, neg_src], axis=0)
		all_dst = tf.concat([pos_dst, neg_dst], axis=0)

		labels = tf.concat([
			tf.ones_like(src, dtype=tf.float32),
			tf.zeros_like(neg_src, dtype=tf.float32),
		], axis=0)
		labels = tf.expand_dims(labels, axis=-1)  # [N, 1]

		sample_weight = tf.concat([
			pos_w,
			tf.ones_like(neg_src, dtype=tf.float32),
		], axis=0)  # [N]

		return all_src, all_dst, labels, sample_weight

	def _weighted_mean(self, values, sample_weight):
		values = tf.cast(values, tf.float32)
		sample_weight = tf.cast(sample_weight, tf.float32)
		denom = tf.reduce_sum(sample_weight) + 1e-8
		return tf.reduce_sum(values * sample_weight) / denom

	def _manual_weighted_bce(self, labels, logits, sample_weight):
		per_example_loss = tf.nn.sigmoid_cross_entropy_with_logits(
			labels=labels,
			logits=logits,
		)  # [N, 1]

		per_example_loss = tf.squeeze(per_example_loss, axis=-1)  # [N]
		return self._weighted_mean(per_example_loss, sample_weight)

	def _full_item_logits(self, src):
		src_emb = self.scorer.item_embedding(src)
		item_emb = self.scorer.item_embedding.embeddings
		item_bias = tf.squeeze(self.scorer.item_bias.embeddings, axis=-1)
		logits = tf.matmul(src_emb, item_emb, transpose_b=True)
		logits = tf.nn.bias_add(logits, item_bias)
		return logits

	def _sampled_softmax_loss(self, src, pos_dst, pos_w):
		labels = tf.cast(tf.expand_dims(pos_dst, axis=-1), tf.int64)
		sampled_values = tf.random.uniform_candidate_sampler(
			true_classes=labels,
			num_true=1,
			num_sampled=self.num_sampled,
			unique=True,
			range_max=self.num_items,
		)
		per_example_loss = tf.nn.sampled_softmax_loss(
			weights=self.scorer.item_embedding.embeddings,
			biases=tf.squeeze(self.scorer.item_bias.embeddings, axis=-1),
			labels=labels,
			inputs=self.scorer.item_embedding(src),
			num_sampled=self.num_sampled,
			num_classes=self.num_items,
			sampled_values=sampled_values,
			remove_accidental_hits=True,
		)
		return self._weighted_mean(per_example_loss, pos_w)

	def train_step(self, data):
		batch = data
		src = tf.cast(batch["src_item"], tf.int32)
		pos_dst = tf.cast(batch["pos_dst_item"], tf.int32)
		pos_w = tf.cast(batch["pos_weight"], tf.float32)

		if self.loss_type == "bce":
			all_src, all_dst, labels, sample_weight = self._make_pos_neg_batch(src, pos_dst, pos_w)

			with tf.GradientTape() as tape:
				logits = self.scorer(
					{"src_item": all_src, "dst_item": all_dst},
					training=True,
				)

				loss = self._manual_weighted_bce(
					labels=labels,
					logits=logits,
					sample_weight=sample_weight,
				)

				if self.scorer.losses:
					loss += tf.add_n(self.scorer.losses)

			grads = tape.gradient(loss, self.scorer.trainable_variables)
			self.optimizer.apply_gradients(zip(grads, self.scorer.trainable_variables))

			probs = tf.sigmoid(logits)
			self.train_loss_tracker.update_state(loss)
			self.train_metric.update_state(labels, probs, sample_weight=sample_weight)
		else:
			with tf.GradientTape() as tape:
				loss = self._sampled_softmax_loss(src, pos_dst, pos_w)

				if self.scorer.losses:
					loss += tf.add_n(self.scorer.losses)

			grads = tape.gradient(loss, self.scorer.trainable_variables)
			self.optimizer.apply_gradients(zip(grads, self.scorer.trainable_variables))

			full_logits = self._full_item_logits(src)
			self.train_loss_tracker.update_state(loss)
			self.train_metric.update_state(pos_dst, full_logits, sample_weight=pos_w)

		return {
			"loss": self.train_loss_tracker.result(),
			self.metric_name: self.train_metric.result(),
		}

	def test_step(self, data):
		batch = data
		src = tf.cast(batch["src_item"], tf.int32)
		pos_dst = tf.cast(batch["pos_dst_item"], tf.int32)
		pos_w = tf.cast(batch["pos_weight"], tf.float32)

		if self.loss_type == "bce":
			all_src, all_dst, labels, sample_weight = self._make_pos_neg_batch(src, pos_dst, pos_w)

			logits = self.scorer(
				{"src_item": all_src, "dst_item": all_dst},
				training=False,
			)

			loss = self._manual_weighted_bce(
				labels=labels,
				logits=logits,
				sample_weight=sample_weight,
			)

			if self.scorer.losses:
				loss += tf.add_n(self.scorer.losses)

			probs = tf.sigmoid(logits)
			self.val_loss_tracker.update_state(loss)
			self.val_metric.update_state(labels, probs, sample_weight=sample_weight)
		else:
			logits = self._full_item_logits(src)
			per_example_loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
				labels=pos_dst,
				logits=logits,
			)
			loss = self._weighted_mean(per_example_loss, pos_w)

			if self.scorer.losses:
				loss += tf.add_n(self.scorer.losses)

			self.val_loss_tracker.update_state(loss)
			self.val_metric.update_state(pos_dst, logits, sample_weight=pos_w)

		return {
			"loss": self.val_loss_tracker.result(),
			self.metric_name: self.val_metric.result(),
		}


class RestoreBestValLossWeights(tf.keras.callbacks.Callback):
	def __init__(self, monitor: str = "val_loss", min_delta: float = 0.0):
		super().__init__()
		self.monitor = monitor
		self.min_delta = float(min_delta)
		self.best_value = np.inf
		self.best_epoch = None
		self.best_weights = None

	def on_epoch_end(self, epoch, logs=None):
		logs = logs or {}
		current = logs.get(self.monitor)
		if current is None:
			return

		current = float(current)
		if current < (self.best_value - self.min_delta):
			self.best_value = current
			self.best_epoch = int(epoch) + 1
			self.best_weights = self.model.get_weights()

	def on_train_end(self, logs=None):
		if self.best_weights is not None:
			self.model.set_weights(self.best_weights)


# ------------------------------------------------------------
# 10) Train function
# ------------------------------------------------------------
def train_item_item_model(
		edges_df: pd.DataFrame,
		embedding_dim: int = 64,
		batch_size: int = 4096,
		epochs: int = 10,
		negatives_per_positive: int = 3,
		lr: float = 3e-4,
		weight_decay: float = 1e-4,
		filter_false_negatives: bool = False,
		max_resample_attempts: int = 3,
		val_frac: float = 0.2,

		negative_sampling_mode: str = "popular",
		negative_sampling_power: float = 0.75,
		loss_type: str = "sampled_softmax",
		num_sampled: Optional[int] = None,
		metric_topk: int = 10,
		early_stopping_patience: Optional[int] = 5,
		early_stopping_min_delta: float = 0.0,
):
	train_edges_df, val_edges_df = train_val_split_edges(
		edges_df=edges_df,
		val_frac=val_frac,
		seed=SEED,
	)

	# Build vocabulary from full graph so train/val share same ids
	item2idx, idx2item = build_item_vocab(edges_df)
	num_items = len(item2idx)

	train_ds = make_positive_tf_dataset(
		edges_df=train_edges_df,
		item2idx=item2idx,
		batch_size=batch_size,
		shuffle=True,
	)

	val_ds = make_positive_tf_dataset(
		edges_df=val_edges_df,
		item2idx=item2idx,
		batch_size=batch_size,
		shuffle=False,
	)

	neg_probs = None
	positive_neighbor_sets = None
	if loss_type == "bce":
		# Usually build negative distribution from train only
		neg_probs = build_negative_sampling_distribution(
			train_edges_df,
			item2idx,
			mode=negative_sampling_mode,
			power=negative_sampling_power,
		)

		if filter_false_negatives:
			positive_neighbor_sets = build_positive_neighbor_sets(train_edges_df, item2idx)

	scorer = ItemItemEncoder(
		num_items=num_items,
		embedding_dim=embedding_dim,
	)

	model = BatchNegativeSamplingModel(
		scorer=scorer,
		num_items=num_items,
		neg_probs=neg_probs,
		negatives_per_positive=negatives_per_positive,
		positive_neighbor_sets=positive_neighbor_sets,
		filter_false_negatives=filter_false_negatives,
		max_resample_attempts=max_resample_attempts,
		loss_type=loss_type,
		num_sampled=num_sampled,
		metric_topk=metric_topk,
	)

	steps_per_epoch = max(1, int(np.ceil(len(train_edges_df) / batch_size)))

	lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
		initial_learning_rate=lr,
		decay_steps=steps_per_epoch,
		decay_rate=weight_decay,
		staircase=True,
	)

	optimizer = tf.keras.optimizers.Adam(
		learning_rate=lr_schedule,
	)

	model.compile(
		optimizer=optimizer,
		run_eagerly=False,  # set True temporarily if debugging
	)

	callbacks = []
	best_val_loss_callback = RestoreBestValLossWeights(
		monitor="val_loss",
		min_delta=early_stopping_min_delta,
	)
	callbacks.append(best_val_loss_callback)

	if early_stopping_patience is not None:
		callbacks.append(
			tf.keras.callbacks.EarlyStopping(
				monitor="val_loss",
				mode="min",
				patience=int(early_stopping_patience),
				min_delta=early_stopping_min_delta,
				restore_best_weights=False,
				verbose=1,
			)
		)

	history = model.fit(
		train_ds,
		validation_data=val_ds,
		epochs=epochs,
		callbacks=callbacks,
		verbose=1,
	)

	if best_val_loss_callback.best_epoch is None:
		val_losses = history.history.get("val_loss", [])
		if val_losses:
			best_idx = int(np.argmin(val_losses))
			best_epoch = best_idx + 1
			best_val_loss = float(val_losses[best_idx])
		else:
			best_epoch = None
			best_val_loss = None
	else:
		best_epoch = best_val_loss_callback.best_epoch
		best_val_loss = float(best_val_loss_callback.best_value)

	return {
		"model": model,
		"scorer": scorer,
		"item2idx": item2idx,
		"idx2item": idx2item,
		"history": history,
		"best_epoch": best_epoch,
		"best_val_loss": best_val_loss,
		"train_edges_df": train_edges_df,
		"val_edges_df": val_edges_df,
	}


# ------------------------------------------------------------
# 11) Embeddings + retrieval
# ------------------------------------------------------------
def get_item_embeddings(model, normalize=True):
	if hasattr(model, "scorer"):
		emb = model.scorer.item_embedding.get_weights()[0]
	else:
		emb = model.item_embedding.get_weights()[0]

	if normalize:
		emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
	return emb


def get_item_biases(model):
	if hasattr(model, "scorer"):
		bias = model.scorer.item_bias.get_weights()[0]
	else:
		bias = model.item_bias.get_weights()[0]
	return bias.reshape(-1)


def get_ann_item_vectors(model):
	emb = get_item_embeddings(model, normalize=False)
	bias = get_item_biases(model)
	return np.hstack([emb, bias[:, None]])


def get_ann_query_vector_for_item(anchor_item, model, item2idx: dict):
	if anchor_item not in item2idx:
		return None

	emb = get_item_embeddings(model, normalize=False)
	anchor_idx = item2idx[anchor_item]
	return np.concatenate([emb[anchor_idx], np.array([1.0], dtype=emb.dtype)])


def get_ann_query_vector_for_cart(cart_items, model, item2idx: dict):
	valid_idxs = [item2idx[x] for x in cart_items if x in item2idx]
	if not valid_idxs:
		return None

	emb = get_item_embeddings(model, normalize=False)
	cart_vec = emb[valid_idxs].mean(axis=0)
	return np.concatenate([cart_vec, np.array([1.0], dtype=emb.dtype)])


def recommend_for_item(
		anchor_item,
		model,
		item2idx: dict,
		idx2item: dict,
		top_k: int = 20,
		exclude_self: bool = True,
		normalize: bool = True,
):
	if anchor_item not in item2idx:
		return pd.DataFrame(columns=["product_id", "score"])

	emb = get_item_embeddings(model, normalize=normalize)
	anchor_idx = item2idx[anchor_item]
	scores = emb @ emb[anchor_idx]

	if exclude_self:
		scores[anchor_idx] = -1e9

	top_k = min(top_k, len(scores))
	top_idx = np.argpartition(-scores, range(top_k))[:top_k]
	top_idx = top_idx[np.argsort(-scores[top_idx])]

	return pd.DataFrame({
		"product_id": [idx2item[i] for i in top_idx],
		"score": scores[top_idx],
	})


def recommend_for_item_train_score(
		anchor_item,
		model,
		item2idx: dict,
		idx2item: dict,
		top_k: int = 20,
		exclude_self: bool = True,
):
	if anchor_item not in item2idx:
		return pd.DataFrame(columns=["product_id", "score"])

	ann_item_vecs = get_ann_item_vectors(model)
	query = get_ann_query_vector_for_item(anchor_item, model, item2idx)
	anchor_idx = item2idx[anchor_item]
	scores = ann_item_vecs @ query

	if exclude_self:
		scores[anchor_idx] = -1e9

	top_k = min(top_k, len(scores))
	top_idx = np.argpartition(-scores, range(top_k))[:top_k]
	top_idx = top_idx[np.argsort(-scores[top_idx])]

	return pd.DataFrame({
		"product_id": [idx2item[i] for i in top_idx],
		"score": scores[top_idx],
	})


def recommend_for_cart(
		cart_items,
		model,
		item2idx: dict,
		idx2item: dict,
		top_k: int = 20,
		normalize: bool = True,
):
	valid_idxs = [item2idx[x] for x in cart_items if x in item2idx]
	if not valid_idxs:
		return pd.DataFrame(columns=["product_id", "score"])

	emb = get_item_embeddings(model, normalize=normalize)
	cart_vec = emb[valid_idxs].mean(axis=0)
	cart_vec = cart_vec / (np.linalg.norm(cart_vec) + 1e-12)

	scores = emb @ cart_vec

	for i in valid_idxs:
		scores[i] = -1e9

	top_k = min(top_k, len(scores))
	top_idx = np.argpartition(-scores, range(top_k))[:top_k]
	top_idx = top_idx[np.argsort(-scores[top_idx])]

	return pd.DataFrame({
		"product_id": [idx2item[i] for i in top_idx],
		"score": scores[top_idx],
	})


def recommend_for_cart_train_score(
		cart_items,
		model,
		item2idx: dict,
		idx2item: dict,
		top_k: int = 20,
):
	valid_idxs = [item2idx[x] for x in cart_items if x in item2idx]
	if not valid_idxs:
		return pd.DataFrame(columns=["product_id", "score"])

	ann_item_vecs = get_ann_item_vectors(model)
	query = get_ann_query_vector_for_cart(cart_items, model, item2idx)
	scores = ann_item_vecs @ query

	for i in valid_idxs:
		scores[i] = -1e9

	top_k = min(top_k, len(scores))
	top_idx = np.argpartition(-scores, range(top_k))[:top_k]
	top_idx = top_idx[np.argsort(-scores[top_idx])]

	return pd.DataFrame({
		"product_id": [idx2item[i] for i in top_idx],
		"score": scores[top_idx],
	})

In [104]:
# import os
# import clickhouse_connect
#
# CLICKHOUSE_CONFIG = {
# 	"host": os.getenv("CH_HOST", "172.30.36.13"),
# 	"port": int(os.getenv("CH_PORT", "8123")),
# 	"username": os.getenv("CH_USER", "default"),
# 	"password": os.getenv("CH_PASSWORD", "192837465AhurA!@#"),
# 	"database": os.getenv("CH_DATABASE", "KF"),
# 	"secure": os.getenv("CH_SECURE", "false").lower() == "true",
# }
#
# client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
#
# activity_type_ids = (1, 29)
# interval = 360
# query_orders = f'''
# SELECT ms.activity_type_id AS activity_type_id,
#         oi.order_id AS order_id,
#         pp.category_level2_id AS category_id
# FROM KF.fact_order_items oi
#             JOIN KF.plane_products pp
#                 ON oi.shop_product_id = pp.shop_product_id AND
#                 order_date BETWEEN today() - {interval} AND yesterday() AND
#                 is_gross = 1 AND
#                 product_id != 0 AND
#                 items_jet_discount = 0
#             JOIN KF.dim_merchant_shops ms
#                 ON ms.merchant_shop_id = pp.merchant_shop_id AND
#                 activity_type_id IN {activity_type_ids} AND
#                 merchant_shop_polygon_id IN (66, 67, 69, 71, 76, 86, 94) AND
#                 category_id NOT IN (
#                 0,
#                 82
#                 )
# GROUP BY ms.activity_type_id, oi.order_id, pp.category_level2_id
# '''
#
# df_baskets = client.query_df(query_orders)
# client.close()

In [105]:
# query_categories = f'''
# SELECT * FROM KF.dim_categories
# '''
#
# df_categories = client.query_df(query_categories)
# client.close()

In [106]:
# df_baskets.to_csv('./sample_data/baskets.csv', index=False)
# df_categories.to_csv('./sample_data/categories.csv', index=False)

In [107]:
df_baskets = pd.read_csv('./sample_data/baskets.csv')
df_categories = pd.read_csv('./sample_data/categories.csv')

In [108]:
min_support = int(df_baskets['order_id'].nunique() * 0.0025)
min_support

1865

In [109]:
df_baskets[df_baskets['category_id'] != 81]

,activity_type_id,order_id,category_id
0,1,20210125,33
1,1,21371882,3122
2,1,15527599,3134
3,1,22991964,3106
4,1,14402225,3091
...,...,...,...
1945392,29,13376124,37
1945393,1,14260533,3097
1945394,1,17480479,70
1945395,1,25023659,70


In [110]:
# Build graph
edges_df, item_freq = build_copurchase_graph(
	df=df_baskets,
	order_col="order_id",
	item_col="category_id",
	min_item_freq=min_support,
	min_items_per_order=2,
	max_items_per_order=100,
	weight_mode="lift_shrink",
	topk_neighbors=10,
)

print("\nGraph edges sample:")
edges_df.head(10)


Graph edges sample:


,src_item,dst_item,weight,cosine,lift,shrink,lift_shrink,ppmi,confidence,co-occurance
0,33,36,1.195413,0.078872,1.199991,0.996185,1.195413,0.182314,0.022017,2611
1,33,3082,1.208902,0.107314,1.211427,0.997916,1.208902,0.191799,0.040374,4788
2,33,3087,1.259997,0.051374,1.272054,0.990521,1.259997,0.240633,0.008812,1045
3,33,137,1.301838,0.189554,1.302775,0.999281,1.301838,0.264497,0.117135,13891
4,33,3121,1.349287,0.139461,1.351148,0.998623,1.349287,0.300955,0.061135,7250
5,33,31,1.397339,0.181479,1.398517,0.999158,1.397339,0.335413,0.100017,11861
6,33,3088,1.404745,0.107235,1.408161,0.997575,1.404745,0.342284,0.034683,4113
7,33,32,1.448492,0.077399,1.455479,0.995199,1.448492,0.375335,0.017480,2073
8,33,3083,1.488818,0.124931,1.491643,0.998106,1.488818,0.399878,0.044439,5270
9,33,37,1.601261,0.267789,1.601971,0.999557,1.601261,0.471235,0.190117,22546


In [111]:
# edges_df['weight'] = edges_df['lift_shrink']
edges_df.dtypes

src_item         object
dst_item         object
weight          float64
cosine          float64
lift            float64
shrink          float64
lift_shrink     float64
ppmi            float64
confidence      float64
co-occurance      int64
dtype: object

In [118]:
artifacts = train_item_item_model(
	edges_df=edges_df,
	embedding_dim=512,
	batch_size=64,
	epochs=100,
	lr=4e-3,
	weight_decay=0.998,
	val_frac=0.1,
	loss_type="sampled_softmax",
	num_sampled=256,
	metric_topk=5,
	early_stopping_patience=10,
	early_stopping_min_delta=1e-4,
)
model = artifacts["model"]
item2idx = artifacts["item2idx"]
idx2item = artifacts["idx2item"]

Epoch 1/100
13/13 [==============================] - 1s 25ms/step - loss: 4.4877 - top5_acc: 0.2570 - val_loss: 4.4151 - val_top5_acc: 0.0849
Epoch 2/100
13/13 [==============================] - 0s 9ms/step - loss: 4.0205 - top5_acc: 0.4829 - val_loss: 4.0812 - val_top5_acc: 0.1616
Epoch 3/100
13/13 [==============================] - 0s 9ms/step - loss: 3.5138 - top5_acc: 0.5116 - val_loss: 3.7999 - val_top5_acc: 0.2069
Epoch 4/100
13/13 [==============================] - 0s 9ms/step - loss: 3.1896 - top5_acc: 0.5035 - val_loss: 3.6870 - val_top5_acc: 0.2669
Epoch 5/100
13/13 [==============================] - 0s 10ms/step - loss: 3.0805 - top5_acc: 0.4934 - val_loss: 3.6620 - val_top5_acc: 0.2908
Epoch 6/100
13/13 [==============================] - 0s 9ms/step - loss: 3.0211 - top5_acc: 0.5023 - val_loss: 3.6359 - val_top5_acc: 0.2446
Epoch 7/100
13/13 [==============================] - 0s 9ms/step - loss: 2.9940 - top5_acc: 0.4984 - val_loss: 3.5785 - val_top5_acc: 0.3135
Epoch 8/100

In [119]:
df_categories = df_categories[df_categories['type'] == 'leaf']
df_categories['category_id'] = df_categories['category_id'].astype(str)

In [120]:
df_baskets.groupby(['category_id'])['order_id'].count().sort_values(ascending=False)

category_id
69      148052
33      131928
70       83363
66       72881
37       66588
         ...  
1082         3
3151         2
3642         2
3127         1
212          1
Name: order_id, Length: 130, dtype: int64

In [122]:
item_id = "3104"  # 88
print(f"Recommendations for item = '{item_id}'")
recoms = recommend_for_item_train_score(
	anchor_item=item_id,
	model=model,
	item2idx=item2idx,
	idx2item=idx2item,
	top_k=10,
	exclude_self=False
)

recoms.merge(df_categories, left_on='product_id', right_on='category_id', how='left')

Recommendations for item = '3104'


,product_id,score,category_id,status,type,title
0,3104,5.535345,3104,active,leaf,رب
1,86,4.724488,86,active,leaf,حبوبات و سویا
2,3100,4.607955,3100,active,leaf,ماکارونی، رشته و لازانیا
3,85,4.592880,85,active,leaf,برنج
4,3101,4.450581,3101,active,leaf,روغن آشپزی و سالاد
5,3103,4.346204,3103,active,leaf,روغن سرخ کردنی
6,3134,4.314091,3134,active,leaf,تن ماهی
7,3133,4.096004,3133,active,leaf,کنسرو سبزیجات و حبوبات
8,88,3.980866,88,active,leaf,شکر، قند و نبات
9,38,3.745835,38,active,leaf,ادویه و افزودنی


In [116]:
edges_df[(edges_df['src_item'] == item_id)].merge(df_categories, left_on='dst_item', right_on='category_id', how='left').sort_values(by='weight',
                                                                                                                                     ascending=False).head(
	50)

,src_item,dst_item,weight,cosine,lift,shrink,lift_shrink,ppmi,confidence,co-occurance,category_id,status,type,title
9,3104,3100,3.024278,0.104969,3.040849,0.994550,3.024278,1.112137,0.167770,1825,3100,active,leaf,ماکارونی، رشته و لازانیا
8,3104,86,3.021249,0.057751,3.076583,0.982014,3.021249,1.123820,0.050193,546,86,active,leaf,حبوبات و سویا
7,3104,3134,2.762925,0.067963,2.796134,0.988124,2.762925,1.028238,0.076485,832,3134,active,leaf,تن ماهی
6,3104,3146,2.449249,0.044761,2.510176,0.975728,2.449249,0.920353,0.036955,402,3146,active,leaf,آبلیمو، سرکه و آبغوره
5,3104,3101,2.443012,0.060915,2.475369,0.986928,2.443012,0.906390,0.069406,755,3101,active,leaf,روغن آشپزی و سالاد
4,3104,85,2.368036,0.026200,2.542156,0.931507,2.368036,0.933012,0.012502,136,85,active,leaf,برنج
3,3104,38,2.293561,0.070226,2.314936,0.990766,2.293561,0.839382,0.098639,1073,38,active,leaf,ادویه و افزودنی
2,3104,3133,2.248652,0.043560,2.302836,0.976471,2.248652,0.834142,0.038150,415,3133,active,leaf,کنسرو سبزیجات و حبوبات
1,3104,3128,2.218509,0.032985,2.312117,0.959514,2.218509,0.838164,0.021787,237,3128,active,leaf,کیسه زباله
0,3104,3103,2.200774,0.050125,2.239726,0.982609,2.200774,0.806353,0.051940,565,3103,active,leaf,روغن سرخ کردنی


In [117]:
print("\nRecommendations for cart = ['phone', 'case']")
recoms = recommend_for_cart_train_score(
	cart_items=["3097", "74", "3144"],
	model=model,
	item2idx=item2idx,
	idx2item=idx2item,
	top_k=10,
)

recoms.merge(df_categories, left_on='product_id', right_on='category_id', how='left')


Recommendations for cart = ['phone', 'case']


,product_id,score,category_id,status,type,title
0,3073,3.849307,3073,active,leaf,پنیر آشپزی
1,3133,3.771492,3133,active,leaf,کنسرو سبزیجات و حبوبات
2,62,3.700304,62,active,leaf,غذای نیمه آماده منجمد
3,3136,3.553755,3136,active,leaf,سبزیجات و صیفی‌‌جات منجمد
4,61,3.419678,61,active,leaf,نان پیتزا و خمیر
5,40,3.177061,40,active,leaf,ترشی و شوری
6,39,3.138591,39,active,leaf,سس
7,63,2.899306,63,active,leaf,کنسرو و غذای آماده مصرف
8,101,2.802966,101,active,leaf,نوشیدنی انرژی‌زا
9,3091,2.793383,3091,active,leaf,آبمیوه
